        # 02 資格取得と転職（RQ2・RQ3）

        - RQ2：**資格取得は生涯所得を増やすか**（費用・取得年齢・回収期間）
        - RQ3：**何歳で転職すると生涯所得が最大になるか**（全探索）

> 架空のデータによる**教育用シミュレーション**です。実際の賃金統計ではなく、将来の所得の予測でもありません。金額は**万円**、税・社会保険料を引く前の**額面**です。

## 準備：実験環境（最初に必ず実行）

F1 リポジトリの `.py`（`scenarios.py`・`income_analysis.py`・`optimization.py` など）を import できるように、作業ディレクトリ（F1 のルート）を自動で探して設定します。

| 実行場所 | 準備 |
| --- | --- |
| ローカルの Jupyter / VSCode | `notebooks/` の中で開けば、そのまま動きます |
| Colab（方法A：GitHub から clone） | 下のセルの `REPO_URL` に**自分たちのリポジトリの URL** を入れる |
| Colab（方法B：ZIP をアップロード） | F1 フォルダを ZIP にして `/content` にアップロードしておく（またはセルの実行中にアップロード） |

> `textbook/`（教科書のコード）は公開リポジトリに含めない約束です。`textbook/` がないと `optimization.py` の import が失敗します。

In [ ]:
#@title 準備（F1 の .py を読み込む）
import os, sys, glob, zipfile, subprocess
from pathlib import Path

# ---- 方法A：GitHub から clone する場合だけ設定する（URL は自分たちのものを入れる）----
REPO_URL = ""                                  # 例："https://github.com/<team>/<repo>.git"（空なら clone しない）
REPO_SUBDIR = "f1-career-income-optimization"  # リポジトリの中の F1 フォルダ（ルートが F1 なら ""）

IN_COLAB = "google.colab" in sys.modules


def is_f1_root(p):
    return (Path(p) / "career_model.py").exists() and (Path(p) / "optimization.py").exists()


def find_f1_root():
    for p in [Path.cwd(), Path.cwd().parent]:          # ローカル：notebooks/ の中、または F1 のルート
        if is_f1_root(p):
            return p.resolve()
    if not IN_COLAB:
        return None
    if REPO_URL:                                        # Colab 方法A
        dest = Path("/content/repo")
        if not dest.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
        cand = dest / REPO_SUBDIR if REPO_SUBDIR else dest
        if is_f1_root(cand):
            return cand
    zips = glob.glob("/content/*.zip")                  # Colab 方法B
    if not zips:
        from google.colab import files
        print("F1 フォルダの ZIP を選んでアップロードしてください")
        zips = ["/content/" + name for name in files.upload()]
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            zf.extractall("/content/f1_upload")
    for hit in glob.glob("/content/f1_upload/**/career_model.py", recursive=True):
        if is_f1_root(Path(hit).parent):
            return Path(hit).parent
    return None


F1_ROOT = find_f1_root()
if F1_ROOT is None:
    raise RuntimeError("F1 のフォルダが見つかりません。上の説明（方法A／方法B）を確認してください。")
os.chdir(F1_ROOT)
if str(F1_ROOT) not in sys.path:
    sys.path.insert(0, str(F1_ROOT))
if not (F1_ROOT / "textbook" / "max.py").exists():
    print("⚠️ textbook/ がありません。optimization.py の import が失敗します。")

# ---- 既存の .py（計算の本体）と、発表用のグラフ ----
import matplotlib.pyplot as plt
import career_model as cm
import scenarios as sc
import income_analysis as ia
import optimization as op
import presentation_charts as pc

pc.setup_style()                      # 発表用に文字を大きめにする（pc.setup_style(1.3) でさらに大きく）
careers, settings = sc.load_scenarios()
named = {s["name"]: s for s in careers}
RATE = settings.get("discount_rate", ia.DEFAULT_DISCOUNT_RATE)

SAVE_FIGURES = False                  # True にすると、スライド用の PNG を notebooks/figures/ に保存する
FIG_DIR = "notebooks/figures"

def show(fig, name):
    """グラフを表示し、SAVE_FIGURES = True なら PNG も保存する"""
    if SAVE_FIGURES:
        print("saved:", pc.save(fig, f"{FIG_DIR}/{name}.png"))
    plt.show()

print("F1_ROOT:", F1_ROOT)
print("キャリア:", [s["label"] for s in careers], " 割引率:", RATE)

## RQ2：資格取得の仮定

- 取得する年に費用を払い、**翌年から**毎年 年収 + `salary_increase` 万円
- Core では**必ず合格**する（成功確率は Advanced A）
- 比べる相手：**同じキャリアで資格を取らない場合**

In [ ]:
cert = named["certificate"]
no_cert = sc.with_changes(cert, certificate=None, name="certificate_base", label="No certificate")
print("資格：", cert["certificate"])
p = ia.payback_age(cert, no_cert)
print(f"回収年齢 {p['payback_age']} 歳、64 歳時点の差 {p['final_gap']:+,.0f} 万円")
show(pc.plot_gap_vs_baseline(cert, no_cert, "RQ2: certificate vs no certificate (cumulative gap)"),
     "rq2_certificate_gap")

## RQ2：何歳で取るかで、効果はどう変わるか

In [ ]:
cert_ages = list(range(22, 64))
gains, paybacks = [], []
for age in cert_ages:
    s = sc.with_changes(cert, **{"certificate.age": age})
    gains.append(ia.cumulative_net_income(s) - ia.cumulative_net_income(no_cert))
    paybacks.append(ia.payback_age(s, no_cert)["payback_age"])

fig, ax = plt.subplots()
ax.bar(cert_ages, gains, color=[pc.color_of("certificate") if g > 0 else pc.NEUTRAL for g in gains])
ax.axhline(0, color=pc.TEXT, lw=1)
ax.set_xlabel("age when the certificate is obtained")
ax.set_ylabel("net gain until 64 [10k JPY]")
ax.set_title("RQ2: the earlier, the larger the gain (gray: not recovered)")
show(fig, "rq2_certificate_age")
last_ok = max((a for a, g in zip(cert_ages, gains) if g > 0), default=None)
print("元が取れる（差がプラスになる）最も遅い取得年齢：", last_ok)

## RQ3：転職年齢の全探索

- 候補：22〜60 歳（$K = 39$ 通り）。各候補で 47 年分を計算 → **$O(KT)$**
- 最大値：教科書の `max_of`（第2章）で最大値、`seq_search`（第3章）でその位置
- 同点なら**最も若い年齢**
- 破線は「転職しない」場合（候補には含まれていない）

In [ ]:
jc = named["job_change"]
first, last = settings.get("switch_age_range", [22, 60])
rows = op.evaluate_switch_ages(jc, op.switch_age_candidates(jc, first, last), RATE)
best = op.optimal_switch_ages(rows)
stay = ia.evaluate(sc.with_changes(jc, job_change=None), RATE)
show(pc.plot_switch_age_search(rows, best, stay, "RQ3: job-change age (new salary 420, fictional)"),
     "rq3_switch_search")
for key, name in op.CRITERIA.items():
    print(f"{name:22}: 最適な転職年齢 {best[key]['switch_age']} 歳（{best[key]['value']:,.0f} 万円）")

## RQ3：なぜ「早いほど良い」になるのか

Core のモデルは「**転職後の初年度年収は、年齢・経験によらず 420 万円**」と仮定しています。

- 早く転職するほど、高い年収（と高い昇給率）の期間が長くなる
- 遅く転職すると、それまでに昇給した年収が 420 万円を上回り、**転職で年収が下がる**

→ 結果は**仮定から出てきたもの**。「早く転職すべき」という結論ではありません。

## RQ3：条件を変えると、評価基準で答えが分かれる

転職後の年収 250 万円・昇給率 2.5%・転職費用 100 万円の場合（Experiment 3 と同じ条件）

In [ ]:
jc2 = sc.with_changes(jc, **{"job_change.new_salary": 250, "job_change.growth_rate": 0.025, "job_change.cost": 100})
rows2 = op.evaluate_switch_ages(jc2, op.switch_age_candidates(jc2, first, last), RATE)
best2 = op.optimal_switch_ages(rows2)
show(pc.plot_switch_age_search(rows2, best2, stay, "RQ3: new salary 250, cost 100 (fictional)"),
     "rq3_switch_search_low_salary")
for key, name in op.CRITERIA.items():
    print(f"{name:22}: {best2[key]['switch_age']} 歳")
print(f"転職しない場合の生涯所得 {stay['gross']:,.0f} 万円 vs 最適な転職 {best2['gross']['value']:,.0f} 万円")

## 自分たちの条件で探索する

In [ ]:
# TODO：転職後の年収・昇給率・費用・候補の範囲を変えて再実行しよう
my_jc = sc.with_changes(jc, **{"job_change.new_salary": 380, "job_change.growth_rate": 0.03, "job_change.cost": 50})
my_rows = op.evaluate_switch_ages(my_jc, op.switch_age_candidates(my_jc, 25, 55), RATE)
my_best = op.optimal_switch_ages(my_rows)
print({k: v["switch_age"] for k, v in my_best.items()})
# TODO：「転職しない」を候補に加える。転職後の年収を経験年数で変える（scenarios.py の TODO 3）

## 考察（チームで記入）

- **TODO**：資格を「何歳までに」取れば元が取れるか。費用・年収増加を変えるとどうなるか
- **TODO**：最適な転職年齢を決めている仮定はどれか。現実的な仮定に変えると結果はどう変わるか
- **TODO**：生涯所得・累積純収入・NPV で最適解が違う条件と、その理由
- **TODO**：資格の不合格、転職先の年収の不確実性を入れると？（Advanced A・E）

## 参考文献

- 柴田望洋『新・明解 Pythonで学ぶアルゴリズムとデータ構造（第2版）』SBクリエイティブ（第2章 最大値、第3章 線形探索、第6章 挿入ソート）
- F1 リポジトリの `README.md`（モデルの仮定、計算量、モデルの限界と倫理的配慮）
- データ：`data/sample_careers.json`（この教材のための**架空のデータ**）